In [ ]:
'''
Evaluate tagged corpus on 100 topics and some statistics 
'''

In [ ]:
%%capture
!pip install ijson
!pip install python-terrier
import pyterrier as pt
if not pt.started():
  pt.init()

import pandas as pd
df = pd.read_csv('/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/tag-essays-fasttext.csv')


In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles = []
title_ids = []
nums = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
topics = pd.DataFrame()
topics['qid'] = nums
topics['query'] = titles


In [ ]:
from nltk.tokenize import RegexpTokenizer
tokenizer = RegexpTokenizer(r'\w+')

qrels_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

docnos = qrels.docno

from nltk.tokenize import RegexpTokenizer
tokenizer = RegexpTokenizer(r'\w+')

df = df.fillna('')
df['tagged'] = df['claims'] + " " + df['premises']

path_targer = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/'
pd_indexer = pt.DFIndexer(path_targer + "pd_index_essays-fasttext-100")
indexref = pd_indexer.index(df["tagged"], df["docno"])

index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

res

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval

In [ ]:
res_copy = res.copy()
res_copy = res_copy[res_copy['docno'].isin(docnos)]
res_copy.head(10)

eval = pt.Utils.evaluate(res_copy,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval

In [ ]:
import numpy as np
tagged_df = pd.read_csv('/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/tag-essays-fasttext.csv')
tagged_df.replace(np.nan, '', inplace = True)
df = pd.read_csv('/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/corpus.csv')

In [ ]:
all_claims = tagged_df['claims']
all_premises = tagged_df['premises']
text_claim_and_premise = []
for claim, premise in zip(all_claims, all_premises):
    text_claim_and_premise.append(claim + " " + premise)

In [ ]:
df_claim_and_premise = pd.DataFrame()
df_claim_and_premise['docno'] = tagged_df['docno']
df_claim_and_premise['text'] = text_claim_and_premise
df_claim_and_premise

In [ ]:
len_orig = []
len_tag = []

i = 0
for tag_text, orig_text in zip(df_claim_and_premise['text'], df['text']):
    len_orig.append(len(tokenizer.tokenize(orig_text)))
    len_tag.append(len(tokenizer.tokenize(tag_text)))
    if(len(tokenizer.tokenize(orig_text)) == 0):
        print(i)
        print(orig_text)
        print(tag_text)
        print('-------')
        break
    i += 1

In [ ]:
lens = np.array([len_orig, len_tag])
print(f'Mean of original lenghtes in tokens: {np.mean(len_orig)}')
print(f'Mean of tagged lenghtes in tokens: {np.mean(len_tag)}')

print(f'Variance of original lenghtes in tokens: {np.var(len_orig)}')
print(f'Variance of tagged lenghtes in tokens: {np.var(len_tag)}')

print(f'Standard deviation of original lenghtes in tokens: {np.std(len_orig)}')
print(f'Standard deviation of tagged lenghtes in tokens: {np.std(len_tag)}')

print(f'Median of original lenghtes in tokens: {np.median(len_orig)}')
print(f'Median of tagged lenghtes in tokens: {np.median(len_tag)}')

print(f'Max of original lenghtes in tokens: {np.max(len_orig)}')
print(f'Max of tagged lenghtes in tokens: {np.max(len_tag)}')

print(f'Min of original lenghtes in tokens: {np.min(len_orig)}')
print(f'Min of tagged lenghtes in tokens: {np.min(len_tag)}')
